# Bài tập Pandas: DataFrame và CSV (Bài 1–8)

Notebook dựa trên các thao tác trong `06-Examples.ipynb`: tạo DataFrame, thêm/sửa cột, xử lý dữ liệu thiếu, đọc/ghi CSV, sắp xếp, lọc bằng nhiều điều kiện, `head`/`tail`, `nsmallest`/`nlargest` và thống kê `mean`/`min`/`max`.

**Chạy trên Google Colab:** tải notebook này lên Colab → `Runtime > Run all`. Khi được hỏi, tải cùng lúc `participant_list.csv` và `nyc_weather.csv` đã được cung cấp. Chạy các ô theo thứ tự từ trên xuống. Bài 5 tạo ra `sorted_participant_list.csv` trong vùng Files của Colab.

## Chuẩn bị dữ liệu và thư viện

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

def get_csv_path(filename):
    # Trên Colab: file upload nằm trong /content. Khi chạy cục bộ: có thể nằm trong upload/.
    for path in [Path(filename), Path('/content') / filename, Path('upload') / filename]:
        if path.is_file():
            return path
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(f'Không thấy {filename}. Hãy đặt file cạnh notebook.') from exc
    print(f'Hãy tải lên {filename} (có thể chọn cả 2 CSV một lần).')
    files.upload()
    path = Path('/content') / filename
    if not path.is_file():
        path = Path(filename)
    if not path.is_file():
        raise FileNotFoundError(f'Chưa tải lên {filename}.')
    return path

participant_path = get_csv_path('participant_list.csv')
weather_path = get_csv_path('nyc_weather.csv')
print('Dữ liệu đã sẵn sàng:', participant_path.name, 'và', weather_path.name)

## Bài 1 — Tạo hai DataFrame

DataFrame thời tiết bên dưới là **bảng mẫu năm 2017 trong đề**; các bài 6–8 đọc file thời tiết riêng (tháng 1/2016). `np.nan` biểu thị ô bị thiếu.

In [ ]:
df_weather_sample = pd.DataFrame(
    {
        'temperature': [32, 35, 28, 24, 32, 31],
        'windspeed': [6, 7, 2, 7, 4, 2],
        'event': ['Rain', 'Sunny', 'Snow', 'Snow', 'Rain', 'Sunny'],
    },
    index=['1/1/2017', '1/2/2017', '1/3/2017',
           '1/4/2017', '1/5/2017', '1/6/2017'],
)

df_people = pd.DataFrame(
    {
        'Name': ['Tom', 'Jack', 'Steve', np.nan],
        'Age': [28.0, 34.0, np.nan, 42.0],
        'City': ['London', np.nan, 'LA', 'Newyork'],
    },
    index=['rank1', 'rank2', 'rank3', 'rank4'],
)
print('DataFrame thời tiết mẫu:')
display(df_weather_sample)
print('DataFrame nhân sự:')
display(df_people)

## Bài 2 — Thêm Gross Salary, Tax, Net Salary

`Net Salary = Gross Salary - Tax`. Tạo bản sao để giữ nguyên bảng gốc của Bài 1.

In [ ]:
df_salary = df_people.copy()
df_salary['Gross Salary'] = [3000, 4000, 2500, 4200]
df_salary['Tax'] = [300, 350, 200, 400]
df_salary['Net Salary'] = df_salary['Gross Salary'] - df_salary['Tax']
display(df_salary)

## Bài 3 — Cập nhật và điền ô thiếu

Tăng mỗi tuổi thêm 1 và `Gross Salary` thêm 200. Sau khi điền các ô thiếu theo bảng kết quả của đề, tính lại `Net Salary`. Dùng `.loc` để ghi từng ô rõ ràng.

In [ ]:
df_final = df_salary.copy()
df_final['Age'] = df_final['Age'] + 1
df_final['Gross Salary'] = df_final['Gross Salary'] + 200

# Tuổi của Steve chưa có ở Bài 2; bảng đích của đề cho kết quả là 29.
df_final.loc['rank3', 'Age'] = 29.0
df_final.loc['rank2', 'City'] = 'Vienna'
df_final.loc['rank4', 'Name'] = 'Alice'
df_final['Net Salary'] = df_final['Gross Salary'] - df_final['Tax']
display(df_final)

## Bài 4 — Chọn cột, lấy 3 hàng đầu và 2 hàng cuối

In [ ]:
df_salary_view = df_final[['Name', 'Gross Salary', 'Tax', 'Net Salary']].copy()
print('DataFrame mới:')
display(df_salary_view)
print('3 hàng đầu:')
display(df_salary_view.head(3))
print('2 hàng cuối:')
display(df_salary_view.tail(2))

## Bài 5 — Đọc, sắp xếp người tham gia và xuất CSV

File `participant_list.csv` đã được cung cấp cùng bài. Đọc `SĐT` dưới dạng chuỗi để giữ nguyên định dạng số điện thoại; `.shape` cho biết `(số hàng, số cột)`. Sắp xếp theo **toàn bộ chuỗi Họ và tên**, đặt lại STT từ 1, rồi lưu UTF-8 có BOM để Excel hiển thị tiếng Việt.

In [ ]:
participants = pd.read_csv(participant_path, encoding='utf-8-sig', dtype={'SĐT': 'string'})
print('Kích thước (hàng, cột):', participants.shape)
display(participants)

sorted_participants = participants.sort_values('Họ và tên', kind='stable').reset_index(drop=True)
sorted_participants['STT'] = range(1, len(sorted_participants) + 1)
sorted_participants = sorted_participants[participants.columns]
sorted_csv_path = Path('sorted_participant_list.csv')
sorted_participants.to_csv(sorted_csv_path, index=False, encoding='utf-8-sig')
print('Danh sách sau sắp xếp:')
display(sorted_participants)
print('Đã lưu:', sorted_csv_path.resolve())

## Bài 6 — Thống kê 10 ngày đầu tháng 1

Lọc tháng 1 từ cột ngày `EST`, sắp xếp theo thời gian rồi lấy 10 ngày đầu. Các chỉ số tính riêng cho `Temperature` và `Humidity` bằng `.agg()`.

In [ ]:
weather = pd.read_csv(weather_path, encoding='utf-8-sig')
weather['Date'] = pd.to_datetime(weather['EST'], format='%m/%d/%Y')

january_first_10 = (
    weather.loc[weather['Date'].dt.month.eq(1), ['EST', 'Date', 'Temperature', 'Humidity']]
    .sort_values('Date')
    .head(10)
    .drop(columns='Date')
    .reset_index(drop=True)
)
print('Nhiệt độ và độ ẩm của 10 ngày đầu:')
display(january_first_10)
print('Trung bình, nhỏ nhất, lớn nhất:')
display(january_first_10[['Temperature', 'Humidity']].agg(['mean', 'min', 'max']))

## Bài 7 — Lọc ngày theo điều kiện và tính tầm nhìn

Điều kiện là **(nhiệt độ > 30 VÀ độ ẩm > 50) HOẶC (tốc độ gió ≥ 10)**. Phải đặt ngoặc quanh từng so sánh khi dùng `&` và `|` trong pandas. Giá trị tốc độ gió bị thiếu không thỏa điều kiện `≥ 10`.

In [ ]:
condition = (
    ((weather['Temperature'] > 30) & (weather['Humidity'] > 50))
    | (weather['WindSpeedMPH'] >= 10)
)
selected_days = weather.loc[
    condition, ['EST', 'Temperature', 'Humidity', 'WindSpeedMPH', 'VisibilityMiles']
].copy()
print('Số ngày được chọn:', len(selected_days))
display(selected_days)
print('Tầm nhìn trung bình, lớn nhất, nhỏ nhất:')
display(selected_days['VisibilityMiles'].agg(['mean', 'max', 'min']).to_frame('VisibilityMiles'))

## Bài 8 — 10 ngày lạnh nhất và 10 ngày nóng nhất

Chọn 10 hàng có `Temperature` nhỏ/lớn nhất bằng `nsmallest()` và `nlargest()`, sau đó tính **trung bình độ ẩm của chính các ngày đó**. Nếu nhiệt độ hòa nhau ở ranh giới, pandas giữ thứ tự xuất hiện trong file.

In [ ]:
coldest_10 = weather.nsmallest(10, 'Temperature')[['EST', 'Temperature', 'Humidity']]
hottest_10 = weather.nlargest(10, 'Temperature')[['EST', 'Temperature', 'Humidity']]
print('10 ngày nhiệt độ thấp nhất:')
display(coldest_10)
print('Độ ẩm trung bình (10 ngày thấp nhất):', coldest_10['Humidity'].mean())
print('10 ngày nhiệt độ cao nhất:')
display(hottest_10)
print('Độ ẩm trung bình (10 ngày cao nhất):', hottest_10['Humidity'].mean())

### Kết quả để tự đối chiếu với dữ liệu đi kèm

- Bài 5: kích thước `(8, 6)`.
- Bài 6: trung bình nhiệt độ `36.4`, độ ẩm `51.0`; nhiệt độ min/max `20/50`; độ ẩm min/max `33/77`.
- Bài 7: `16` ngày; tầm nhìn trung bình `8.625`, min/max `1/10`.
- Bài 8: độ ẩm trung bình của 10 ngày lạnh nhất `48.6`; của 10 ngày nóng nhất `57.7`.

Các đơn vị được giữ theo tên cột và giá trị của file CSV nguồn.